
# 04_02 Optimize performance with cross-validation and hyperparameter search

In this notebook, you will use cross-validation and automated hyperparameter search to improve a classification model without overfitting.


## Load the breast cancer dataset

In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer(as_frame=True)
df = data.frame.copy()

X = df[data.feature_names]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)

X_train.shape, X_test.shape


((398, 30), (171, 30))

## Fit a baseline Random Forest model

In [2]:

from sklearn.ensemble import RandomForestClassifier

baseline_model = RandomForestClassifier(random_state=1)
baseline_model.fit(X_train, y_train)

baseline_test_score = baseline_model.score(X_test, y_test)
baseline_test_score


0.9473684210526315

## Evaluate with cross-validation

In [3]:

from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(baseline_model, X_train, y_train, cv=5)
cv_scores, cv_scores.mean()


(array([0.9625    , 0.9125    , 0.975     , 0.96202532, 0.96202532]),
 np.float64(0.9548101265822785))

## Set up a hyperparameter grid for search

In [4]:

from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5]
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    n_jobs=-1,
    scoring="accuracy"
)

grid_search


GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [None, 5, 10],
                         'min_samples_split': [2, 5],
                         'n_estimators': [50, 100, 200]},
             scoring='accuracy')

## Run the grid search

In [5]:

grid_search.fit(X_train, y_train)

grid_search.best_params_, grid_search.best_score_


({'max_depth': None, 'min_samples_split': 5, 'n_estimators': 50},
 np.float64(0.9548101265822784))

## Evaluate the best model on the test set

In [6]:

best_model = grid_search.best_estimator_
best_test_score = best_model.score(X_test, y_test)
best_test_score


0.9473684210526315

## Inspect hyperparameter search results

In [7]:

results_df = pd.DataFrame(grid_search.cv_results_)
results_df = results_df.sort_values("mean_test_score", ascending=False)

results_df[[
    "param_n_estimators", "param_max_depth", "param_min_samples_split", "mean_test_score"
]].head()


,param_n_estimators,param_max_depth,param_min_samples_split,mean_test_score
15,50,10,5,0.954810
3,50,None,5,0.954810
6,50,5,2,0.952278
0,50,None,2,0.952247
12,50,10,2,0.952247
